### 0. 프로젝트 Clone

In [ ]:
import os
if not os.path.isdir("/content/OSAP_proj1/src"):
    !git clone https://github.com/sjjeon0925/OSAP_proj1.git /content/OSAP_proj1
os.chdir("/content/OSAP_proj1")
print("CWD:", os.getcwd())

### 1. Install Dependencies

In [ ]:
!pip install wandb fvcore pyyaml pycocotools

### 2. WandB Login

In [ ]:
import wandb
# Colab 실행 시 프롬프트에서 API 키 입력 (wandb.ai/settings)
wandb.login()

### 2-1. Google Drive Mount (체크포인트 영구 저장용)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. 프로젝트 루트 폴더로 이동

In [ ]:
import os
# 현재 작업 폴더가 notebooks라면 상위 폴더(루트)로 이동
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Current Working Directory:', os.getcwd())

### 3-1. MS-COCO 압축 해제 (세션 시작 시 1회 실행)

In [ ]:
import os, zipfile, yaml

with open("src/config/config.yaml") as f:
    config = yaml.safe_load(f)

data_cfg  = config["data"]
coco_root = data_cfg["coco_root"]

COCO_DOWNLOADS = [
    (
        "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        data_cfg["coco_zip_annotations"],
        os.path.join(coco_root, "annotations"),
    ),
    (
        "http://images.cocodataset.org/zips/train2017.zip",
        data_cfg["coco_zip_images"],
        os.path.join(coco_root, "train2017"),
    ),
]

for url, zip_path, check_dir in COCO_DOWNLOADS:
    name = os.path.basename(zip_path)
    if os.path.isdir(check_dir):
        print(f"[COCO] Already extracted: {check_dir}")
        continue
    if not os.path.isfile(zip_path):
        print(f"[COCO] Downloading {name} to Drive ...")
        os.makedirs(os.path.dirname(zip_path), exist_ok=True)
        ret = os.system(f'wget -q --show-progress -O "{zip_path}" "{url}"')
        if ret != 0:
            raise RuntimeError(f"Download failed: {url}")
    else:
        print(f"[COCO] zip found on Drive: {name}")
    print(f"[COCO] Extracting {name} ...")
    os.makedirs(coco_root, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(coco_root)
    print(f"[COCO] Done: {check_dir}")

print("[COCO] Ready.")

### 4. Training 실행

In [ ]:
!python src/train.py

### 5. Evaluation 실행

In [ ]:
!python src/eval.py

### 6. Inference & ONNX Export 실행
- `submit/img/` 에 테스트 이미지를 넣고 실행하면 `submit/pred/` 에 PNG 마스크가, `submit/model_structure.onnx` 가 생성됩니다.

In [ ]:
!pip install onnx onnxscript -q
!python src/inference.py